In [7]:
import os

# Either correct the variable …
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.5 pyspark-shell"
)

# … or simply remove it if you’re already passing the package via builder.config:
# os.environ.pop("PYSPARK_SUBMIT_ARGS", None)

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("VehicleLocationDataValidation")
    .master("local[*]")                         # optional but explicit
    .config(
        "spark.jars",
        "/Users/gyauk/codetools/javajdbc/postgresql-42.7.5.jar",
    )
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.5",
    )
    .getOrCreate()
)

print("Spark version:", spark.version)


Spark version: 3.5.5


In [8]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, to_timestamp, to_date, year, current_date, when, regexp_extract, upper, lower, ltrim, rtrim, length, isnull, isnan
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, TimestampType, DoubleType, BooleanType

# Initialize Spark Session
# In a real EMR environment, SparkSession is often pre-configured,
# but for local testing, we need to create it.
spark = SparkSession.builder \
    .appName("VehicleLocationDataValidation") \
    .getOrCreate()
        # .config("spark.driver.extraClassPath", "/path/to/hadoop-aws.jar:/path/to/aws-java-sdk-bundle.jar") \

# --- 1. Define Schemas ---

# Vehicle Data Schema
# We define nullable as True initially, and then use validation to check for nulls
vehicle_schema = StructType([
    StructField("active", IntegerType(), True),
    StructField("vehicle_license_number", StringType(), True),
    StructField("registration_name", StringType(), True),
    StructField("license_type", StringType(), True),
    StructField("expiration_date", StringType(), True), # Read as String for initial validation
    StructField("permit_license_number", StringType(), True),
    StructField("certification_date", StringType(), True), # Read as String for initial validation
    StructField("vehicle_year", IntegerType(), True),
    StructField("base_telephone_number", StringType(), True),
    StructField("base_address", StringType(), True),
    StructField("vehicle_id", StringType(), True),
    StructField("last_update_timestamp", StringType(), True), # Read as String for initial validation
    StructField("brand", StringType(), True),
    StructField("vehicle_type", StringType(), True)
])

# Location Data Schema
location_schema = StructType([
    StructField("location_id", IntegerType(), True),
    StructField("location_name", StringType(), True),
    StructField("address", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("zip_code", StringType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True)
])


In [9]:
# --- 2. Read Data ---

# Local file paths (ensure these CSV files are in your project directory or provide full paths)
local_vehicle_data_path = "/Users/gyauk/Desktop/DataEngineering/phase_2/Lab4/vehicles.csv"
local_location_data_path = "/Users/gyauk/Desktop/DataEngineering/phase_2/Lab4/locations.csv"

# S3 paths (commented out as requested for local testing)
# s3_vehicle_data_path = "s3a://your-s3-bucket-name/vehicle_data.csv"
# s3_location_data_path = "s3a://your-s3-bucket-name/location_data.csv"

print("Reading vehicle data from local file...")
try:
    df_vehicles = spark.read \
        .option("header", "true") \
        .schema(vehicle_schema) \
        .csv(local_vehicle_data_path)
    print("Vehicle data schema after initial read:")
    df_vehicles.printSchema()
    df_vehicles.show(5, truncate=False)
except Exception as e:
    print(f"Error reading local vehicle data: {e}")
    df_vehicles = None



Reading vehicle data from local file...
Vehicle data schema after initial read:
root
 |-- active: integer (nullable = true)
 |-- vehicle_license_number: string (nullable = true)
 |-- registration_name: string (nullable = true)
 |-- license_type: string (nullable = true)
 |-- expiration_date: string (nullable = true)
 |-- permit_license_number: string (nullable = true)
 |-- certification_date: string (nullable = true)
 |-- vehicle_year: integer (nullable = true)
 |-- base_telephone_number: string (nullable = true)
 |-- base_address: string (nullable = true)
 |-- vehicle_id: string (nullable = true)
 |-- last_update_timestamp: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- vehicle_type: string (nullable = true)

+------+----------------------+------------------------------+----------------+---------------+---------------------+------------------+------------+---------------------+----------------------------------------+----------+---------------------+---------+-----

In [10]:
print("\nReading location data from local file...")
try:
    df_locations = spark.read \
        .option("header", "true") \
        .schema(location_schema) \
        .csv(local_location_data_path)
    print("Location data schema after initial read:")
    df_locations.printSchema()
    df_locations.show(5, truncate=False)
except Exception as e:
    print(f"Error reading local location data: {e}")
    df_locations = None



Reading location data from local file...
Location data schema after initial read:
root
 |-- location_id: integer (nullable = true)
 |-- location_name: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zip_code: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)

+-----------+-------------------------------+----------------------------+---------+-----+--------+-----------+----------+
|location_id|location_name                  |address                     |city     |state|zip_code|latitude   |longitude |
+-----------+-------------------------------+----------------------------+---------+-----+--------+-----------+----------+
|2702       |Jackson, Velazquez and Gonzales|3140 Heath Radial Apt. 604  |Modesto  |CA   |94540   |86.25802   |-169.2448 |
|4380       |Bean LLC                       |51144 Patrick Isle Suite 397|Fontana  |CA   |92188  

In [12]:

# --- 3. Basic Validations and Transformations (Vehicle Data) ---

if df_vehicles:
    print("\n--- Validating and Transforming Vehicle Data ---")

    # 3.1. Null Checks and Handling
    print("\nChecking for nulls in critical vehicle columns...")
    critical_vehicle_cols = ["vehicle_license_number", "expiration_date", "vehicle_id", "brand", "vehicle_type"]
    for col_name in critical_vehicle_cols:
        null_count = df_vehicles.filter(col(col_name).isNull()).count()
        if null_count > 0:
            print(f"WARNING: Column '{col_name}' has {null_count} null values.")
            # Example: Drop rows where critical columns are null
            df_vehicles = df_vehicles.na.drop(subset=[col_name])
            print(f"Dropped rows with nulls in '{col_name}'. Remaining rows: {df_vehicles.count()}")

    #  Data Type Conversions and Formatting
    print("\nPerforming data type conversions and formatting for vehicle data...")

    # Convert 'expiration_date' and 'certification_date' to DateType
    # Assuming 'expiration_date' format is DD-MM-YYYY
    # Assuming 'certification_date' format is YYYY-MM-DD
    df_vehicles_transformed = df_vehicles.withColumn("expiration_date", to_date(col("expiration_date"), "dd-MM-yyyy")) \
                                         .withColumn("certification_date", to_date(col("certification_date"), "yyyy-MM-dd")) \
                                         .withColumn("last_update_timestamp", to_timestamp(col("last_update_timestamp"), "dd-MM-yyyy HH:mm:ss"))

    # Validate date conversions (check for nulls after conversion, indicating parse failures)
    df_vehicles_transformed = df_vehicles_transformed.withColumn("expiration_date_valid",
        when(col("expiration_date").isNull(), lit(False)).otherwise(lit(True)))
    df_vehicles_transformed = df_vehicles_transformed.withColumn("certification_date_valid",
        when(col("certification_date").isNull(), lit(False)).otherwise(lit(True)))
    df_vehicles_transformed = df_vehicles_transformed.withColumn("last_update_timestamp_valid",
        when(col("last_update_timestamp").isNull(), lit(False)).otherwise(lit(True)))

    # Handle invalid dates
    invalid_exp_dates = df_vehicles_transformed.filter(col("expiration_date_valid") == False).count()
    if invalid_exp_dates > 0:
        print(f"WARNING: {invalid_exp_dates} invalid 'expiration_date' values detected after conversion.")
    invalid_cert_dates = df_vehicles_transformed.filter(col("certification_date_valid") == False).count()
    if invalid_cert_dates > 0:
        print(f"WARNING: {invalid_cert_dates} invalid 'certification_date' values detected after conversion.")
    invalid_update_ts = df_vehicles_transformed.filter(col("last_update_timestamp_valid") == False).count()
    if invalid_update_ts > 0:
        print(f"WARNING: {invalid_update_ts} invalid 'last_update_timestamp' values detected after conversion.")


    # Validate 'active' column: should be 0 or 1
    df_vehicles_transformed = df_vehicles_transformed.withColumn("active_valid",
        when((col("active") == 0) | (col("active") == 1), lit(True)).otherwise(lit(False)))
    invalid_active = df_vehicles_transformed.filter(col("active_valid") == False).count()
    if invalid_active > 0:
        print(f"WARNING: {invalid_active} invalid 'active' values (not 0 or 1).")
        # Example: Set invalid 'active' values to null or a default
        df_vehicles_transformed = df_vehicles_transformed.withColumn("active",
            when(col("active_valid"), col("active")).otherwise(lit(None).cast(IntegerType())))


    # Trim whitespace from string columns
    string_cols_to_trim = ["registration_name", "license_type", "permit_license_number", "base_address", "brand", "vehicle_type"]
    for col_name in string_cols_to_trim:
        df_vehicles_transformed = df_vehicles_transformed.withColumn(col_name, ltrim(rtrim(col(col_name))))

    # Example: Standardize 'license_type' to uppercase
    df_vehicles_transformed = df_vehicles_transformed.withColumn("license_type", upper(col("license_type")))

    # Example: Extract State from base_address if not already available separately
    # # This is a very basic regex, might need refinement for real-world addresses
    # df_vehicles_transformed = df_vehicles_transformed.withColumn("base_state_extracted",
    #     regexp_extract(col("base_address"), r'.*\s([A-Z]{2})\s\d{5}$', 1)) # Assumes CA in "SAN FRANCISCO CA 94158"

    print("\nSchema after vehicle transformations:")
    df_vehicles_transformed.printSchema()
    print("\nSample Vehicle Data after transformations:")
    df_vehicles_transformed.select("active", "vehicle_license_number", "expiration_date", "certification_date", "vehicle_year", "base_address", "license_type", "active_valid", "expiration_date_valid").show(5, truncate=False)



--- Validating and Transforming Vehicle Data ---

Checking for nulls in critical vehicle columns...

Performing data type conversions and formatting for vehicle data...

Schema after vehicle transformations:
root
 |-- active: integer (nullable = true)
 |-- vehicle_license_number: string (nullable = true)
 |-- registration_name: string (nullable = true)
 |-- license_type: string (nullable = true)
 |-- expiration_date: date (nullable = true)
 |-- permit_license_number: string (nullable = true)
 |-- certification_date: date (nullable = true)
 |-- vehicle_year: integer (nullable = true)
 |-- base_telephone_number: string (nullable = true)
 |-- base_address: string (nullable = true)
 |-- vehicle_id: string (nullable = true)
 |-- last_update_timestamp: timestamp (nullable = true)
 |-- brand: string (nullable = true)
 |-- vehicle_type: string (nullable = true)
 |-- expiration_date_valid: boolean (nullable = false)
 |-- certification_date_valid: boolean (nullable = false)
 |-- last_update_tim

In [ ]:

# ---  Basic Validations and Transformations (Location Data) ---

if df_locations:
    print("\n--- Validating and Transforming Location Data ---")

    #  Null Checks and Handling
    print("\nChecking for nulls in critical location columns...")
    critical_location_cols = ["location_id", "location_name", "address", "city", "state", "zip_code", "latitude", "longitude"]
    for col_name in critical_location_cols:
        null_count = df_locations.filter(col(col_name).isNull()).count()
        if null_count > 0:
            print(f"WARNING: Column '{col_name}' has {null_count} null values.")
            # Example: Drop rows where critical columns are null
            df_locations = df_locations.na.drop(subset=[col_name])
            print(f"Dropped rows with nulls in '{col_name}'. Remaining rows: {df_locations.count()}")

    # Data Type Conversions and Formatting
    print("\nPerforming data type conversions and formatting for location data...")

    # Ensure latitude and longitude are within valid ranges
    df_locations_transformed = df_locations.withColumn("latitude_valid",
        when((col("latitude") >= -90) & (col("latitude") <= 90), lit(True)).otherwise(lit(False)))
    df_locations_transformed = df_locations_transformed.withColumn("longitude_valid",
        when((col("longitude") >= -180) & (col("longitude") <= 180), lit(True)).otherwise(lit(False)))

    invalid_lat = df_locations_transformed.filter(col("latitude_valid") == False).count()
    if invalid_lat > 0:
        print(f"WARNING: {invalid_lat} invalid 'latitude' values detected (out of range).")
    invalid_lon = df_locations_transformed.filter(col("longitude_valid") == False).count()
    if invalid_lon > 0:
        print(f"WARNING: {invalid_lon} invalid 'longitude' values detected (out of range).")

    # Trim whitespace from string columns
    trim_cols = ["location_name", "address", "city", "state", "zip_code"]
    for col_name in trim_cols:
        df_locations_transformed = df_locations_transformed.withColumn(col_name, ltrim(rtrim(col(col_name))))

    #Standardize 'state' to uppercase
    df_locations_transformed = df_locations_transformed.withColumn("state", upper(col("state")))

    # Validate zip_code length (assuming 5 digits)
    df_locations_transformed = df_locations_transformed.withColumn("zip_code_valid",
        when(length(col("zip_code")) == 5, lit(True)).otherwise(lit(False)))
    invalid_zip = df_locations_transformed.filter(col("zip_code_valid") == False).count()
    if invalid_zip > 0:
        print(f"WARNING: {invalid_zip} invalid 'zip_code' values (not 5 digits).") 


    print("\nSchema after location transformations:")
    df_locations_transformed.printSchema()
    print("\nSample Location Data after transformations:")
    df_locations_transformed.select("location_id", "location_name", "address", "city", "state", "zip_code", "latitude", "longitude", "latitude_valid", "longitude_valid", "zip_code_valid").show(5, truncate=False)

# Stop Spark Session
spark.stop()
print("\nSpark Session stopped.")
